# UNSW-NB15 Dataset Understanding

Goal: understand the pre-split UNSW-NB15 CSV files before building the first binary intrusion detection model.

Binary target:

- `label = 0`: normal traffic
- `label = 1`: attack traffic

Columns excluded from the MVP model:

- `id`: row identifier
- `attack_cat`: multiclass category that directly describes the attack type

In [ ]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

WORKSPACE_ROOT = PROJECT_ROOT.parent
DATA_DIR = WORKSPACE_ROOT / "UNSW-NB15 dataset" / "CSV Files" / "Training and Testing Sets"

TRAIN_PATH = DATA_DIR / "UNSW_NB15_training-set.csv"
TEST_PATH = DATA_DIR / "UNSW_NB15_testing-set.csv"

TRAIN_PATH, TEST_PATH

In [ ]:
train = pd.read_csv(TRAIN_PATH, encoding="utf-8-sig")
test = pd.read_csv(TEST_PATH, encoding="utf-8-sig")

train.shape, test.shape

In [ ]:
train.head()

In [ ]:
pd.DataFrame({
    "column": train.columns,
    "dtype": [train[col].dtype for col in train.columns],
    "missing": [train[col].isna().sum() for col in train.columns],
    "unique_values": [train[col].nunique(dropna=False) for col in train.columns],
})

## Binary Target Distribution

In [ ]:
target_summary = pd.concat(
    [
        train["label"].value_counts().sort_index().rename("train_count"),
        train["label"].value_counts(normalize=True).sort_index().mul(100).round(2).rename("train_pct"),
        test["label"].value_counts().sort_index().rename("test_count"),
        test["label"].value_counts(normalize=True).sort_index().mul(100).round(2).rename("test_pct"),
    ],
    axis=1,
)

target_summary

## Attack Categories

`attack_cat` is useful for analysis, but it must be excluded from the first binary model input.

In [ ]:
attack_summary = pd.DataFrame({
    "count": train["attack_cat"].value_counts(),
    "pct": train["attack_cat"].value_counts(normalize=True).mul(100).round(2),
})

attack_summary

## Feature Groups

In [ ]:
excluded_columns = ["id", "attack_cat", "label"]
feature_data = train.drop(columns=excluded_columns)

categorical_features = feature_data.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_features = [col for col in feature_data.columns if col not in categorical_features]

len(numeric_features), numeric_features, len(categorical_features), categorical_features

In [ ]:
for column in categorical_features:
    print(column)
    print(train[column].value_counts().head(20))
    print()

## Data Quality Checks

In [ ]:
quality_summary = {
    "train_missing_values": int(train.isna().sum().sum()),
    "test_missing_values": int(test.isna().sum().sum()),
    "train_duplicate_rows": int(train.duplicated().sum()),
    "test_duplicate_rows": int(test.duplicated().sum()),
}

quality_summary

## First Modeling Decision

The first pipeline should:

- use `label` as the target
- drop `id` and `attack_cat`
- one-hot encode `proto`, `service`, and `state`
- use the remaining columns as numeric features
- evaluate precision, recall, F1-score, ROC-AUC, and confusion matrix

Accuracy alone is not enough because the classes are imbalanced.